# GenAI DataOps and Vector Platform Pipeline

Advanced notebook for ingestion pipelines, SQL/NoSQL/vector storage design, and API-ready retrieval services.

## Scope

1. Run multi-source document ingestion with validation.
2. Build SQL metadata store, JSON document lake, and vector index.
3. Measure retrieval throughput and consistency.
4. Generate API/testing/CI integration artifacts.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import math
import random
import re
import sqlite3
from collections import Counter

random.seed(59)
ARTIFACT_DIR = Path("artifacts/genai_pipeline")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
sources = []
domains = ["incident", "cicd", "rag", "security", "cloud"]
for i in range(220):
    domain = domains[i % len(domains)]
    body = f"{domain} policy note {i}. Must satisfy platform controls and observability guardrails."
    sources.append({"doc_id": f"SRC-{i:04d}", "domain": domain, "body": body, "owner": f"team-{i % 9}"})

assert len(sources) == 220
assert len({s["doc_id"] for s in sources}) == 220
print({"records": len(sources), "domains": sorted(set(s["domain"] for s in sources))})

{'records': 220, 'domains': ['cicd', 'cloud', 'incident', 'rag', 'security']}


In [3]:
conn = sqlite3.connect(":memory:")
cur = conn.cursor()
cur.execute("CREATE TABLE docs (doc_id TEXT PRIMARY KEY, domain TEXT, owner TEXT, body TEXT)")
for s in sources:
    cur.execute("INSERT INTO docs VALUES (?, ?, ?, ?)", (s["doc_id"], s["domain"], s["owner"], s["body"]))
conn.commit()
cur.execute("SELECT domain, COUNT(*) FROM docs GROUP BY domain ORDER BY domain")
print("sql_counts=", cur.fetchall())

json_store = {s["doc_id"]: s for s in sources}
(ARTIFACT_DIR / "dataops_doc_store.json").write_text(json.dumps(json_store, indent=2), encoding="utf-8")

sql_counts= [('cicd', 44), ('cloud', 44), ('incident', 44), ('rag', 44), ('security', 44)]


42176

In [4]:
TOKEN_RE = re.compile(r"[a-z0-9]+")


def tokenize(text: str):
    return TOKEN_RE.findall(text.lower())


def vectorize(text: str):
    return Counter(tokenize(text))


def cosine(a: Counter, b: Counter) -> float:
    terms = set(a) & set(b)
    num = sum(a[t] * b[t] for t in terms)
    den = math.sqrt(sum(v * v for v in a.values())) * math.sqrt(sum(v * v for v in b.values()))
    return 0.0 if den == 0 else num / den


vector_index = {s["doc_id"]: vectorize(s["body"]) for s in sources}


def search(query: str, top_k: int = 5):
    q = vectorize(query)
    rows = [(doc_id, cosine(q, vec)) for doc_id, vec in vector_index.items()]
    rows.sort(key=lambda x: x[1], reverse=True)
    return rows[:top_k]


demo = search("incident controls observability", top_k=5)
print("top_hits=", demo)

top_hits= [('SRC-0000', 0.5222329678670935), ('SRC-0005', 0.5222329678670935), ('SRC-0010', 0.5222329678670935), ('SRC-0015', 0.5222329678670935), ('SRC-0020', 0.5222329678670935)]


In [5]:
test_queries = [
    {"q": "incident controls", "domain": "incident"},
    {"q": "ci checks and controls", "domain": "cicd"},
    {"q": "vector and rag quality", "domain": "rag"},
]

checks = []
for t in test_queries:
    ranked = search(t["q"], top_k=5)
    domains = [json_store[doc_id]["domain"] for doc_id, _ in ranked]
    checks.append({"query": t["q"], "hit3": int(t["domain"] in domains[:3]), "top_domains": domains[:3]})

hit3 = sum(c["hit3"] for c in checks) / len(checks)
consistency_report = {"hit3": round(hit3, 4), "checks": checks}
(ARTIFACT_DIR / "dataops_consistency_report.json").write_text(json.dumps(consistency_report, indent=2), encoding="utf-8")
print(json.dumps(consistency_report, indent=2))

{
  "hit3": 1.0,
  "checks": [
    {
      "query": "incident controls",
      "hit3": 1,
      "top_domains": [
        "incident",
        "incident",
        "incident"
      ]
    },
    {
      "query": "ci checks and controls",
      "hit3": 1,
      "top_domains": [
        "incident",
        "cicd",
        "rag"
      ]
    },
    {
      "query": "vector and rag quality",
      "hit3": 1,
      "top_domains": [
        "rag",
        "rag",
        "rag"
      ]
    }
  ]
}


In [6]:
fastapi_design = {
    "endpoints": ["/ingest", "/search", "/health", "/ready"],
    "contracts": {
        "/ingest": {"request": {"doc_id": "str", "domain": "str", "body": "str"}, "response": {"status": "ok"}},
        "/search": {"request": {"query": "str", "top_k": "int"}, "response": {"hits": "list"}},
    },
}

pytest_design = {
    "tests": [
        "test_ingest_schema_validation",
        "test_search_returns_ranked_hits",
        "test_search_recall_threshold",
    ]
}

cicd_steps = [
    "ruff check .",
    "pytest -q tests/unit",
    "pytest -q tests/integration",
    "python scripts/run_notebook_tests.py --validate-only",
]

cloud_layout = {
    "azure": ["Blob Storage", "Azure AI Search", "Container Apps", "Azure Monitor"],
    "aws": ["S3", "OpenSearch / Kendra", "ECS Fargate", "CloudWatch"],
}

bundle = {
    "api": fastapi_design,
    "testing": pytest_design,
    "cicd": cicd_steps,
    "cloud": cloud_layout,
}
(ARTIFACT_DIR / "dataops_platform_bundle.json").write_text(json.dumps(bundle, indent=2), encoding="utf-8")
print(json.dumps(bundle, indent=2))

{
  "api": {
    "endpoints": [
      "/ingest",
      "/search",
      "/health",
      "/ready"
    ],
    "contracts": {
      "/ingest": {
        "request": {
          "doc_id": "str",
          "domain": "str",
          "body": "str"
        },
        "response": {
          "status": "ok"
        }
      },
      "/search": {
        "request": {
          "query": "str",
          "top_k": "int"
        },
        "response": {
          "hits": "list"
        }
      }
    }
  },
  "testing": {
    "tests": [
      "test_ingest_schema_validation",
      "test_search_returns_ranked_hits",
      "test_search_recall_threshold"
    ]
  },
  "cicd": [
    "ruff check .",
    "pytest -q tests/unit",
    "pytest -q tests/integration",
    "python scripts/run_notebook_tests.py --validate-only"
  ],
  "cloud": {
    "azure": [
      "Blob Storage",
      "Azure AI Search",
      "Container Apps",
      "Azure Monitor"
    ],
    "aws": [
      "S3",
      "OpenSearch / Kendra",
    